# 02 - Data Cleaning

## Customer Intelligence Platform

This notebook executes the data cleaning pipeline defined in `src/cleaning.py`.
The pipeline removes leakage columns, fixes data types, handles missing values,
and produces a clean dataset for downstream analysis.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.load_data import load_raw
from src.cleaning import clean_data, run_cleaning_pipeline
from src.config import CLEAN_FILE, TARGET
from src.utils import df_summary


## 1. Load Raw Data


In [ ]:
df_raw = load_raw()
df_summary(df_raw, "Raw Data")
print(f"\nColumns: {list(df_raw.columns)}")


## 2. Run Cleaning Pipeline

The cleaning pipeline performs the following steps:
1. Drop 8 leakage/irrelevant columns
2. Drop 5 geographic columns (loaded separately when needed)
3. Convert `Total Charges` from string to numeric
4. Impute 11 missing `Total Charges` values with $0 (new customers with 0 tenure)
5. Verify target variable integrity


In [ ]:
df_clean = clean_data(df_raw)
df_summary(df_clean, "Cleaned Data")


In [ ]:
print(f"Columns retained ({len(df_clean.columns)}):")
for i, col in enumerate(df_clean.columns):
    print(f"  {i+1}. {col} ({df_clean[col].dtype})")


## 3. Validation Checks


In [ ]:
# Check no leakage columns remain
from src.config import LEAKAGE_COLUMNS, GEOGRAPHIC_COLUMNS
leaked = [c for c in LEAKAGE_COLUMNS if c in df_clean.columns]
geo = [c for c in GEOGRAPHIC_COLUMNS if c in df_clean.columns]
print(f"Leakage columns remaining: {leaked or 'None'}")
print(f"Geographic columns remaining: {geo or 'None'}")

# Check Total Charges is numeric
print(f"\nTotal Charges dtype: {df_clean['Total Charges'].dtype}")
print(f"Total Charges nulls: {df_clean['Total Charges'].isnull().sum()}")

# Check target
print(f"\nTarget distribution:")
print(df_clean[TARGET].value_counts())


## 4. Save Cleaned Data


In [ ]:
from src.utils import save_dataframe
save_dataframe(df_clean, CLEAN_FILE)
print(f"Saved to: {CLEAN_FILE}")


## Summary

| Step | Before | After |
|------|--------|-------|
| Rows | 7,043 | 7,043 |
| Columns | 33 | 20 |
| Missing Values | 5,185 | 0 |
| Data Types Fixed | Total Charges (object -> float64) | Done |

The cleaned dataset is saved to `data/interim/telco_clean.csv` and is ready for
exploratory data analysis and feature engineering.
